# 02 — Exploratory Data Analysis (HFBTP)

**Purpose:** characterize the 24,687-configuration response surface: feature/target distributions, workload-performance relationships, correlations (with their limitations), grid structure, outliers, and the throughput-latency trade-off. Every figure is saved to `experiments/figures/`.

**Statistical discipline:** one observation per configuration cell means all plots are descriptive associations, not replicated experiments. No confidence intervals, no causal language, no time-series interpretation.

## Setup

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from fabric_perf.data.hfbtp import HFBTPDatasetLoader
from fabric_perf.utils.helpers import figures_dir, results_dir, save_json

FIGS = figures_dir(); RES = results_dir()
loader = HFBTPDatasetLoader(ROOT / "data" / "raw" / "hfbtp")
df = loader.load_raw()
df.shape

(24687, 8)

## 1. Feature and target distributions

In [2]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
cols = ['Set Transaction Arrival Rate','Actual Transaction Arrival Rate','Block Size','Orderers','Throughput','Avg Latency','Min Latency','Max Latency']
for ax, c in zip(axes.ravel(), cols):
    ax.hist(df[c], bins=50, color='#4C72B0')
    ax.set_title(c, fontsize=9)
fig.suptitle("HFBTP distributions (raw columns)")
fig.tight_layout()
fig.savefig(FIGS / "hfbtp_distributions.png", dpi=150)
plt.show()
print("Throughput min/median/max:", df.Throughput.min(), df.Throughput.median(), df.Throughput.max())
print("AvgLatency min/median/max (s):", df['Avg Latency'].min(), df['Avg Latency'].median(), df['Avg Latency'].max())

Throughput min/median/max: 9.8 95.6 197.3
AvgLatency min/median/max (s): 0.08 0.85 2.67


C:\Users\mailf\AppData\Local\Temp\ipykernel_35140\2824486358.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Workload vs throughput: the saturation shape

In [3]:
fig, ax = plt.subplots(figsize=(9,5))
sub = df[df['Orderers']==3]
sc = ax.scatter(sub['Set Transaction Arrival Rate'], sub['Throughput'], s=6, c=sub['Block Size'], cmap='viridis', alpha=0.6)
ax.plot([0,210],[0,210],'k--',lw=1,label='ideal: committed = offered')
ax.set_xlabel('Set transaction arrival rate (tx/s)'); ax.set_ylabel('Throughput (tx/s)')
ax.set_title('Throughput vs offered load (orderers=3), color = block size')
fig.colorbar(sc, label='Block Size')
ax.legend()
fig.tight_layout(); fig.savefig(FIGS/'hfbtp_throughput_vs_load.png', dpi=150); plt.show()
best = df.groupby('Set Transaction Arrival Rate')['Throughput'].max()
deficit = (best.index.values - best.values)
print("per-level best-throughput deficit vs offered load (every 4th level):")
print(sorted(zip(best.index.values[::4], deficit[::4]), key=lambda t: t[0])[:10])

per-level best-throughput deficit vs offered load (every 4th level):
[(np.float64(10.0), np.float64(0.0)), (np.float64(30.0), np.float64(0.0)), (np.float64(50.0), np.float64(0.0)), (np.float64(70.0), np.float64(0.09999999999999432)), (np.float64(90.0), np.float64(0.0)), (np.float64(110.0), np.float64(0.09999999999999432)), (np.float64(130.0), np.float64(0.5)), (np.float64(150.0), np.float64(0.09999999999999432)), (np.float64(170.0), np.float64(1.0)), (np.float64(190.0), np.float64(2.0))]


C:\Users\mailf\AppData\Local\Temp\ipykernel_35140\2063034697.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS/'hfbtp_throughput_vs_load.png', dpi=150); plt.show()


## 3. Block size and latency: inverted-U plus the 2 s orderer-timeout shelf

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(15,4.5))
sub150 = df[(df['Set Transaction Arrival Rate']==150) & (df['Orderers']==3)]
axes[0].plot(sub150['Block Size'], sub150['Avg Latency'], marker='o', ms=3)
axes[0].set_xlabel('Block Size'); axes[0].set_ylabel('Avg Latency (s)'); axes[0].set_title('Latency vs block size (set=150, ord=3)')
axes[1].hist(df['Max Latency'], bins=80, color='#C44E52')
axes[1].axvline(2.0, color='k', ls='--', lw=1)
axes[1].set_title('Max Latency distribution (~2 s batch-timeout shelf)')
sub9 = df[(df['Orderers']==9)]
axes[2].scatter(sub9['Block Size'], sub9['Throughput'], s=3, c=sub9['Set Transaction Arrival Rate'], cmap='plasma', alpha=0.5)
axes[2].set_xlabel('Block Size'); axes[2].set_ylabel('Throughput (tx/s)'); axes[2].set_title('Throughput vs block (ord=9), color=set rate')
for ax in axes:
    ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIGS/'hfbtp_block_latency_shelf.png', dpi=150); plt.show()
shelf = (df['Max Latency'].between(2.0, 2.35)).mean()
print(f"share of rows with Max Latency in [2.0, 2.35] s: {shelf:.1%} (consistent with the 2 s default batch-timeout boundary)")

share of rows with Max Latency in [2.0, 2.35] s: 55.9% (consistent with the 2 s default batch-timeout boundary)


C:\Users\mailf\AppData\Local\Temp\ipykernel_35140\1228334596.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS/'hfbtp_block_latency_shelf.png', dpi=150); plt.show()


## 4. Correlations — and why they understate non-linearity

In [5]:
corr_cols = ['Set Transaction Arrival Rate','Actual Transaction Arrival Rate','Block Size','Orderers','Throughput','Avg Latency','Min Latency','Max Latency']
cm = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(7.5,6))
im = ax.imshow(cm, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(corr_cols))); ax.set_xticklabels(corr_cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr_cols))); ax.set_yticklabels(corr_cols, fontsize=8)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{cm.iloc[i,j]:.2f}', ha='center', va='center', fontsize=6.5)
fig.colorbar(im, shrink=.8)
fig.suptitle('Pearson correlation (linear only - cannot express threshold/saturation effects)')
fig.tight_layout(); fig.savefig(FIGS/'hfbtp_correlations.png', dpi=150); plt.show()
print("Orderers: r ~ 0 with both targets, but orderers matter strongly under high load (next section) - the linear view is misleading by design.")

Orderers: r ~ 0 with both targets, but orderers matter strongly under high load (next section) - the linear view is misleading by design.


C:\Users\mailf\AppData\Local\Temp\ipykernel_35140\2369552458.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS/'hfbtp_correlations.png', dpi=150); plt.show()


## 5. Orderer-count effects appear only under load

In [6]:
hi = df[df['Set Transaction Arrival Rate']>=180]
lo = df[df['Set Transaction Arrival Rate']<=20]
print("Throughput spread across orderers, low load (<=20):", round(lo.groupby('Orderers')['Throughput'].mean().std(),3))
print("Throughput spread across orderers, high load (>=180):", round(hi.groupby('Orderers')['Throughput'].mean().std(),3))
fig, axes = plt.subplots(1, 2, figsize=(12,4.2), sharey=False)
for ax, (lbl, d) in zip(axes, [('low load (set<=20)', lo), ('high load (set>=180)', hi)]):
    m = d.groupby(['Block Size','Orderers'])['Throughput'].mean().unstack()
    for o in (3,5,7,9):
        ax.plot(m.index, m[o], label=f'{o} orderers')
    ax.set_xlabel('Block Size'); ax.set_ylabel('Throughput (tx/s)'); ax.set_title(lbl); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIGS/'hfbtp_orderer_effect_by_load.png', dpi=150); plt.show()

Throughput spread across orderers, low load (<=20): 0.012
Throughput spread across orderers, high load (>=180): 0.456


C:\Users\mailf\AppData\Local\Temp\ipykernel_35140\632315597.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS/'hfbtp_orderer_effect_by_load.png', dpi=150); plt.show()


## 6. Throughput-latency trade-off

In [7]:
fig, ax = plt.subplots(figsize=(8.5,5.5))
sc = ax.scatter(df['Throughput'], df['Avg Latency'], s=4, c=df['Set Transaction Arrival Rate'], cmap='viridis', alpha=0.5)
fig.colorbar(sc, label='Set arrival rate (tx/s)')
ax.set_xlabel('Throughput (tx/s)'); ax.set_ylabel('Avg Latency (s)'); ax.set_title('Throughput-latency trade-off (descriptive surface)')
ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIGS/'hfbtp_throughput_latency_tradeoff.png', dpi=150); plt.show()
low = df[df['Set Transaction Arrival Rate']<=35]
mid = df[(df['Set Transaction Arrival Rate']>=55) & (df['Set Transaction Arrival Rate']<=100)]
heavy = df[df['Set Transaction Arrival Rate']>=165]
print("mean latency by band: low=%.2fs mid=%.2fs heavy=%.2fs" % (low['Avg Latency'].mean(), mid['Avg Latency'].mean(), heavy['Avg Latency'].mean()))

mean latency by band: low=1.12s mid=0.99s heavy=0.70s


C:\Users\mailf\AppData\Local\Temp\ipykernel_35140\3446050367.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS/'hfbtp_throughput_latency_tradeoff.png', dpi=150); plt.show()


## 7. Unusual observations (flag, never remove)

In [8]:
div = (df['Set Transaction Arrival Rate'] - df['Actual Transaction Arrival Rate']).abs()
flags = {
    'set_vs_actual_divergence_gt_5tx': int((div>5).sum()),
    'max_latency_gt_3s': int((df['Max Latency']>3).sum()),
    'throughput_gt_offered': int((df['Throughput'] > df['Set Transaction Arrival Rate']).sum()),
    'avg_latency_gt_2p5s': int((df['Avg Latency']>2.5).sum()),
}
flags

{'set_vs_actual_divergence_gt_5tx': 20,
 'max_latency_gt_3s': 44,
 'throughput_gt_offered': 0,
 'avg_latency_gt_2p5s': 1}

## 8. Persist the EDA tables

In [9]:
tables = {
    'n_rows': int(len(df)),
    'target_summary': {c: {'min': float(df[c].min()), 'median': float(df[c].median()), 'max': float(df[c].max())} for c in ['Throughput','Avg Latency']},
    'correlation_with_throughput': {c: float(round(df[c].corr(df['Throughput']),3)) for c in ['Set Transaction Arrival Rate','Actual Transaction Arrival Rate','Block Size','Orderers']},
    'correlation_with_avg_latency': {c: float(round(df[c].corr(df['Avg Latency']),3)) for c in ['Set Transaction Arrival Rate','Actual Transaction Arrival Rate','Block Size','Orderers']},
    'flags': flags,
    'interpretation_note': 'Descriptive associations on a single observation per configuration cell; no replication, no timestamps; no causal or temporal inference is permitted from these figures.',
}
save_json(tables, RES / 'hfbtp_eda_summary.json')
print('saved:', RES / 'hfbtp_eda_summary.json')

saved: D:\Reseach_Model\AI-Assisted-Permissioned-Blockchain-Architecture-\experiments\results\hfbtp_eda_summary.json
